In [ ]:
pip install --upgrade battery-data-steward

## Quick Start: Convert A Mixed Arbin Workbook

This workbook contains both time-series data and EIS data:

- time-series sheet: `Channel_4_1`
- EIS sheet: `ACIM_Chan4`

The fastest workflow is to convert each sheet explicitly.

In [ ]:
from pathlib import Path
import battery_data_standard as bds
import polars as pl

input_path = Path("INR21700_M50T_T23_Aging_EIS_SOC20_W8_Channel_4.xlsx")
output_dir = Path("bds_outputs") / "eis_soc20_w8_channel_4"
output_dir.mkdir(parents=True, exist_ok=True)

time_sheet = "Channel_4_1"
eis_sheet = "ACIM_Chan4"

time_csv = output_dir / "channel_4_1.bds.csv"
eis_csv = output_dir / "acim_chan4.eis.csv"

time_report = bds.convert(
    input_path,
    time_csv,
    cycler="arbin",
    sheet=time_sheet,
    current_sign="preserve",
    report_path="auto",
)

eis_report = bds.convert_eis(
    input_path,
    eis_csv,
    sheet=eis_sheet,
    format="csv",
)

print("BDS version:", bds.__version__)
print("Time-series CSV:", time_csv)
print("Time-series valid:", time_report.validation.valid)
print("Time-series rows:", time_report.rows)
print("EIS CSV:", eis_csv)
print("EIS valid:", eis_report.valid)
print("EIS rows:", eis_report.rows)



---



---



# BDS 0.2.2 Mixed Arbin Workbook Walkthrough

This notebook runs `battery-data-steward` on one real Arbin workbook that contains both time-series and EIS worksheets:

`INR21700_M50T_T23_Aging_EIS_SOC20_W8_Channel_4.xlsx`

- Time-series sheet: `Channel_4_1`
- EIS sheet: `ACIM_Chan4`

Dataset citation:

Pozzato, G., Allam, A., & Onori, S. (2023, March 1). *Lithium-ion battery aging dataset based on electric vehicle real-driving profiles*. Retrieved from osf.io/qsabn.

It demonstrates sheet inspection, time-series diagnostics, EIS reading, validation, conversion, report writing, and CLI parity.

## 1. Environment And Input File

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

# load data
repo_root = Path("/content")
input_path = repo_root / "INR21700_M50T_T23_Aging_EIS_SOC20_W8_Channel_4.xlsx"
output_dir = repo_root / "examples" / "output" / "eis_soc20_w8_channel_4"
output_dir.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("Input exists:", input_path.exists())
print("Input:", input_path)
print("Output directory:", output_dir)
assert input_path.exists(), f"Missing input file: {input_path}"

## 2. Use The Latest BDS Package

In [ ]:
import battery_data_standard as bds

print("BDS version:", bds.__version__)
assert bds.__version__ >= "0.2.1", "This notebook expects battery-data-steward 0.2.1 or newer."

## 3. Inspect Workbook Sheets

Before using BDS, inspect the workbook sheets so the selected EIS sheet is visible.

In [ ]:
import pandas as pd

excel_file = pd.ExcelFile(input_path)
sheets = excel_file.sheet_names

print("Sheet count:", len(sheets))
for index, sheet in enumerate(sheets):
    print(f"{index}: {sheet}")

## 4. Detect Data Kind

`detect_kind()` identifies whether the file should be routed as time-series, EIS, unsupported helper content, or unknown.

In [ ]:
kind = bds.detect_kind(input_path)

print("Kind:", kind.kind)
print("Confidence:", kind.confidence)
print("Reason:", kind.reason)
print("Path:", kind.path)

## 5. Run Explain Diagnostics

`bds.explain()` is the 0.2.2 single-file diagnostic entry point. For an EIS workbook, it should identify the EIS route and return a structured diagnostic payload without writing converted data.

In [ ]:
explain_report = bds.explain(input_path)

print(explain_report.to_text())

payload = explain_report.to_dict()
print("\nPayload keys:")
for key in payload:
    print("-", key)

## 6. Save Explain Reports

Write JSON, HTML, Excel, and PDF diagnostic reports for review.

In [ ]:
report_paths = bds.write_explain_reports(
    explain_report,
    output_dir,
    formats=("json", "html", "xlsx", "pdf"),
    stem="eis_soc20_w8_channel_4_explain",
)

for fmt, path in report_paths.items():
    print(fmt, "->", path)

## 7. Review The Diagnostic JSON

Read back the JSON report and show the most important routing fields.

In [ ]:
json_report_path = Path(report_paths["json"])
diagnostic = json.loads(json_report_path.read_text(encoding="utf-8"))

fields = [
    "status",
    "selected_adapter",
    "confidence",
    "sheet",
    "source_columns",
    "canonical_columns",
    "recommended_next_action",
    "error_type",
    "error",
]

for field in fields:
    print(f"{field}:")
    print(diagnostic.get(field))
    print()

## 8. Read The EIS Table

For EIS files, use `read_eis()` instead of the ordinary time-series `read()` path.

In [ ]:
eis_df = bds.read_eis(input_path)

print("Rows:", eis_df.height)
print("Columns:", eis_df.columns)
eis_df.head(10)

## 9. Validate The EIS Table

`validate_eis()` checks the standardized EIS columns and numeric validity.

In [ ]:
eis_validation = bds.validate_eis(eis_df)

print("Valid:", eis_validation.valid)
print("Schema:", eis_validation.schema_version)
print("Rows:", eis_validation.rows)
print("Columns:", eis_validation.columns)

if eis_validation.issues:
    for issue in eis_validation.issues:
        print(issue.to_dict())
else:
    print("No validation issues.")

## 10. Convert EIS To Standard CSV And Parquet

`convert_eis()` writes the standardized EIS table. CSV is easy to inspect; Parquet is useful for analysis pipelines.

In [ ]:
eis_csv = output_dir / "INR21700_M50T_T23_Aging_EIS_SOC20_W8_Channel_4.eis.csv"
eis_parquet = output_dir / "INR21700_M50T_T23_Aging_EIS_SOC20_W8_Channel_4.eis.parquet"

csv_report = bds.convert_eis(input_path, eis_csv, format="csv")
parquet_report = bds.convert_eis(input_path, eis_parquet, format="parquet")

print("CSV:", eis_csv)
print("CSV valid:", csv_report.valid)
print("Parquet:", eis_parquet)
print("Parquet valid:", parquet_report.valid)

## 11. Read Back Converted Outputs

Confirm that the written CSV and Parquet files contain the same standardized columns.

In [ ]:
import polars as pl

csv_df = pl.read_csv(eis_csv)
parquet_df = pl.read_parquet(eis_parquet)

print("CSV shape:", csv_df.shape)
print("Parquet shape:", parquet_df.shape)
print("Same columns:", csv_df.columns == parquet_df.columns)

csv_df.head(10)

## 12. Quick EIS Summary

Summarize the standardized impedance table.

In [ ]:
summary = {
    "rows": eis_df.height,
    "frequency_min_hz": eis_df["Frequency_Hz"].min(),
    "frequency_max_hz": eis_df["Frequency_Hz"].max(),
    "zre_min_ohm": eis_df["Zre_exp_Ohm"].min(),
    "zre_max_ohm": eis_df["Zre_exp_Ohm"].max(),
    "zim_min_ohm": eis_df["Zim_exp_Ohm"].min(),
    "zim_max_ohm": eis_df["Zim_exp_Ohm"].max(),
}

summary

## 13. Nyquist Plot

Plot `Zre_exp_Ohm` against `-Zim_exp_Ohm` for a quick visual check.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

if plt is None:
    print("matplotlib is not installed. Install it if a plot is needed.")
else:
    plot_df = eis_df.select(["Zre_exp_Ohm", "Zim_exp_Ohm"]).to_pandas()

    fig, ax = plt.subplots(figsize=(4, 4), dpi=140)
    ax.plot(plot_df["Zre_exp_Ohm"], -plot_df["Zim_exp_Ohm"], marker="o", linewidth=1)
    ax.set_xlabel("Zre_exp_Ohm")
    ax.set_ylabel("-Zim_exp_Ohm")
    ax.set_title("Nyquist Plot")
    ax.grid(True, alpha=0.25)
    plt.show()

## 14. CLI Parity

The same operations are available through the `bds` command line.

In [ ]:
cli_json = output_dir / "cli_eis_soc20_w8_channel_4_explain.json"
cli_html = output_dir / "cli_eis_soc20_w8_channel_4_explain.html"
cli_xlsx = output_dir / "cli_eis_soc20_w8_channel_4_explain.xlsx"
cli_pdf = output_dir / "cli_eis_soc20_w8_channel_4_explain.pdf"

cmd = [
    sys.executable,
    "-m",
    "battery_data_standard.cli",
    "explain",
    str(input_path),
    "--json",
    str(cli_json),
    "--html",
    str(cli_html),
    "--xlsx",
    str(cli_xlsx),
    "--pdf",
    str(cli_pdf),
]

run = subprocess.run(cmd, text=True, capture_output=True, check=True)

print(run.stdout[:2000])
print("JSON exists:", cli_json.exists())
print("HTML exists:", cli_html.exists())
print("XLSX exists:", cli_xlsx.exists())
print("PDF exists:", cli_pdf.exists())

## 15. Output Inventory

List the generated outputs. These files are written under `examples/output/`, which should stay outside the committed notebook unless intentionally published.

In [ ]:
for path in sorted(output_dir.glob("*")):
    print(path.name, path.stat().st_size)